In [1]:
!uv pip install evaluate

error: No virtual environment found; run `uv venv` to create an environment, or pass `--system` to install into a non-virtual environment


In [4]:
import os
import torch
import evaluate
import pandas as pd

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, Audio

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

In [9]:
MODEL_NAME = "openai/whisper-small"
CSV_PATH = "https://www.kaggle.com/datasets/oleksiymaliovanyy/call-center-transcripts-dataset?select=call_recordings.csv"
AUDIO_DIR = "https://www.kaggle.com/datasets/oleksiymaliovanyy/call-center-transcripts-dataset"
TEXT_COLUMN = "Transcript"
OUTPUT_DIR = "./whisper_customer_support"
LANGUAGE = "english"
TASK = "transcribe"

In [13]:
BATCH_SIZE = 2
LR = 1e-4
EPOCHS = 10

df = pd.read_csv(CSV_PATH,sep=None)

audio_col = None
for col in df.columns:
    if "id" in col.lower() or "file" in col.lower():
        audio_col = col
        break
if audio_col is None:
    df["audio_col"] = ""
    audio_col="audio_col"
df["audio_path"] = df[audio_col].apply(
    lambda x: os.path.join(AUDIO_DIR, f"{x}.wav")
)
dataset = Dataset.from_pandas(df)

dataset = dataset.cast_column(
    "audio_path",
    Audio(sampling_rate=16000)
)

/var/folders/ph/zvqs27md5975ljkl6_b3_s_00000gn/T/ipykernel_4895/2593414490.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support sep=None with delim_whitespace=False; you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(CSV_PATH,sep=None)


In [14]:
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task=TASK
)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE,
    task=TASK
)
model.config.suppress_tokens = []

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [15]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=[
        "q_proj",
        "v_proj"
    ]
)

model = get_peft_model(model, lora_config)

In [23]:
import librosa

def prepare_dataset(batch):

    audio_path = batch["audio_path"]

    audio_array, sampling_rate = librosa.load(
        audio_path,
        sr=16000
    )

    input_features = processor.feature_extractor(
        audio_array,
        sampling_rate=sampling_rate
    ).input_features[0]

    labels = processor.tokenizer(
        batch[TEXT_COLUMN]
    ).input_ids

    batch["input_features"] = input_features
    batch["labels"] = labels

    return batch

dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

train_dataset = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names
)

test_dataset_processed = test_dataset.map(
    prepare_dataset,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

ImportError: To support decoding audio data, please install 'torchcodec'.

In [8]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor
)

In [12]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):

    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = 100 * wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer
    }

In [10]:
!uv pip install jiwer

Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 176ms                                         
Prepared 2 packages in 48ms                                              
Installed 2 packages in 5ms                                 
 + jiwer==4.0.0
 + rapidfuzz==3.14.5


In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH_SIZE,

    learning_rate=LR,

    num_train_epochs=EPOCHS,

    fp16=torch.cuda.is_available(),

    eval_strategy="no",

    save_strategy="epoch",

    logging_steps=1,

    predict_with_generate=True,

    generation_max_length=128,

    remove_unused_columns=False,

    label_names=["labels"],

    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,

    model=model,

    train_dataset=train_dataset,

    data_collator=data_collator,

    processing_class=processor.feature_extractor,

    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("Training complete.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,1.913208
2,1.773038
3,1.890153
4,1.745103
5,1.644840
6,1.593955
7,1.625276
8,1.616727
9,1.569808
10,1.393878


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Training complete.


In [18]:
from peft import PeftModel
FINETUNED_PATH = "./whisper_customer_support"

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
).to(device)

finetuned_base = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

finetuned_model = PeftModel.from_pretrained(
    finetuned_base,
    FINETUNED_PATH
).to(device)

wer_metric = evaluate.load("wer")

base_predictions = []
finetuned_predictions = []
references = []

for sample in test_dataset:

    audio = sample["audio_path"]

    input_features = processor(
        audio["array"],
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():

        pred_ids_base = base_model.generate(
            input_features=input_features
        )
        
        pred_ids_ft = finetuned_model.generate(
            input_features=input_features
        )

    pred_base = processor.batch_decode(
        pred_ids_base,
        skip_special_tokens=True
    )[0]

    pred_ft = processor.batch_decode(
        pred_ids_ft,
        skip_special_tokens=True
    )[0]

    ref = sample["Transcript"]

    base_predictions.append(pred_base)
    finetuned_predictions.append(pred_ft)
    references.append(ref)

base_wer = wer_metric.compute(
    predictions=base_predictions,
    references=references
)

finetuned_wer = wer_metric.compute(
    predictions=finetuned_predictions,
    references=references
)

print(f"Base Whisper WER      : {base_wer * 100:.2f}%")
print(f"Fine-tuned Whisper WER: {finetuned_wer * 100:.2f}%")


# for i in range(len(references)):

#     print(f"\nSample {i+1}")

#     print(f"Reference     : {references[i]}")
#     print(f"Base Whisper  : {base_predictions[i]}")
#     print(f"Fine-tuned    : {finetuned_predictions[i]}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Base Whisper WER      : 6.25%
Fine-tuned Whisper WER: 4.02%
